# TuSimple Testing Notebook

This notebook evaluates a trained LaneNet or LaneNetResNet34 checkpoint on the TuSimple dataset using TuSimple-style lane detection metrics: accuracy, false positive rate, and false negative rate. It also saves predictions in a JSON-lines format similar to the TuSimple submission format.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone or pull the project code into Colab.
from pathlib import Path
import os

REPO = "LaneDetection"
REPO_DIR = Path(f"/content/{REPO}")

if not REPO_DIR.exists():
    !git clone https://github.com/abdullahtapanci/LaneDetection.git /content/LaneDetection
else:
    !cd /content/LaneDetection && git pull

%cd /content/LaneDetection

In [ ]:
# Install missing packages if Colab does not already have them.
!pip -q install scikit-learn tqdm opencv-python-headless

In [ ]:
import sys
import json
import inspect
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

PROJECT_DIR = Path('/content/LaneDetection')
sys.path.insert(0, str(PROJECT_DIR))

from DeepLearningTechnique.src.models.lanenet import LaneNet, LaneNetResNet34
from DeepLearningTechnique.src.postprocess import my_postprocess, draw_lanes
import DeepLearningTechnique.src.config as cfg

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Input size:', cfg.IMAGE_WIDTH, cfg.IMAGE_HEIGHT)

## Configuration

Update the paths below if your Drive folder names are different. `TUSIMPLE_ROOT` should point to the folder that contains `train_set`, `test_set`, and/or TuSimple label JSON files.

In [ ]:
TUSIMPLE_ROOT = Path('/content/drive/MyDrive/Lane_Detection_Project/data/tusimple')
CHECKPOINT_PATH = Path('/content/drive/MyDrive/Lane_Detection_Project/checkpoints/1.0_Draft.pt')
OUTPUT_DIR = Path('/content/drive/MyDrive/Lane_Detection_Project/tusimple_test_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TuSimple labels are JSON-lines files. Common names include label_data_0313.json,
# label_data_0531.json, label_data_0601.json, and test_label.json.
LABEL_FILES = [
    TUSIMPLE_ROOT / 'test_label.json',
    TUSIMPLE_ROOT / 'train_set' / 'label_data_0313.json',
    TUSIMPLE_ROOT / 'train_set' / 'label_data_0531.json',
    TUSIMPLE_ROOT / 'train_set' / 'label_data_0601.json',
]

# Set to None for the full evaluation. Use a small number first to verify paths.
MAX_SAMPLES = None

# Post-processing parameters. These are the same style as the Streamlit app.
POSTPROCESS_KWARGS = dict(
    min_blob_pixels=30,
    lane_probability_threshold=0.95,
    horizon_min_pixels=12,
    horizon_padding=20,
    eps=0.35,
    min_samples=20,
    spatial_weight=0.5,
    clustering_mode='embedding_spatial',
    min_pixels=100,
    poly_degree=2,
)

if not any(path.exists() for path in LABEL_FILES) and TUSIMPLE_ROOT.exists():
    LABEL_FILES = sorted(TUSIMPLE_ROOT.rglob('label_data_*.json')) + sorted(TUSIMPLE_ROOT.rglob('test_label.json'))

print('TuSimple root exists:', TUSIMPLE_ROOT.exists(), TUSIMPLE_ROOT)
print('Checkpoint exists:', CHECKPOINT_PATH.exists(), CHECKPOINT_PATH)
print('Output directory:', OUTPUT_DIR)
print('Existing label files:')
for path in LABEL_FILES:
    if path.exists():
        print(' ', path)

In [ ]:
def read_json_lines(path):
    rows = []
    with open(path, 'r') as file:
        for line in file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_tusimple_annotations(label_files, max_samples=None):
    annotations = []
    for label_file in label_files:
        if not label_file.exists():
            continue
        file_rows = read_json_lines(label_file)
        for row in file_rows:
            row['_label_file'] = str(label_file)
        annotations.extend(file_rows)
        print(f'Loaded {len(file_rows)} rows from {label_file.name}')
    if max_samples is not None:
        annotations = annotations[:max_samples]
    return annotations


annotations = load_tusimple_annotations(LABEL_FILES, MAX_SAMPLES)
print('Total samples selected:', len(annotations))
if annotations:
    print('Example keys:', annotations[0].keys())
    print('Example raw_file:', annotations[0]['raw_file'])

## Model Loading

The checkpoint architecture is detected from the first key in the state dictionary. This follows the same idea used in the Streamlit application.

In [ ]:
def load_lanenet_checkpoint(checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint.get('model_state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint
    first_key = next(iter(state_dict), '')

    if first_key.startswith('encoder.initial_block'):
        model = LaneNet(embedding_dim=4)
        architecture = 'LaneNet'
    else:
        model = LaneNetResNet34(embedding_dim=4, pretrained=False)
        architecture = 'LaneNetResNet34'

    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    epoch = checkpoint.get('epoch') if isinstance(checkpoint, dict) else None
    return model, architecture, epoch


model, architecture, checkpoint_epoch = load_lanenet_checkpoint(CHECKPOINT_PATH, DEVICE)
print('Architecture:', architecture)
print('Checkpoint epoch:', checkpoint_epoch)

## TuSimple Metric Helpers

The following evaluator uses the common TuSimple metric logic: predicted lane curves are sampled at the same `h_samples` as the ground truth, then accuracy, FP, and FN are calculated from matched lane lines.

In [ ]:
class TuSimpleLaneEval:
    pixel_thresh = 20
    point_thresh = 0.85

    @staticmethod
    def get_angle(xs, ys):
        xs = np.asarray(xs, dtype=np.float32)
        ys = np.asarray(ys, dtype=np.float32)
        valid = xs >= 0
        xs = xs[valid]
        ys = ys[valid]
        if len(xs) <= 1:
            return 0.0
        try:
            coefficient = np.polyfit(ys, xs, deg=1)[0]
        except Exception:
            return 0.0
        return float(np.arctan(coefficient))

    @staticmethod
    def line_accuracy(pred, gt, thresh):
        pred = np.asarray(pred, dtype=np.float32)
        gt = np.asarray(gt, dtype=np.float32)
        valid = gt >= 0
        if valid.sum() == 0:
            return 0.0
        correct = np.abs(pred[valid] - gt[valid]) < thresh
        return float(correct.sum() / valid.sum())

    @classmethod
    def evaluate_image(cls, pred_lanes, gt_lanes, h_samples):
        if len(gt_lanes) == 0:
            fp = 1.0 if len(pred_lanes) > 0 else 0.0
            return {'accuracy': 0.0, 'fp': fp, 'fn': 0.0}

        angles = [cls.get_angle(gt_lane, h_samples) for gt_lane in gt_lanes]
        thresholds = [cls.pixel_thresh / max(np.cos(angle), 1e-6) for angle in angles]

        line_accuracies = []
        fn = 0.0
        for gt_lane, threshold in zip(gt_lanes, thresholds):
            accuracies = [cls.line_accuracy(pred_lane, gt_lane, threshold) for pred_lane in pred_lanes]
            max_accuracy = max(accuracies) if accuracies else 0.0
            line_accuracies.append(max_accuracy)
            if max_accuracy < cls.point_thresh:
                fn += 1.0

        fp = 0.0
        for pred_lane in pred_lanes:
            accuracies = [cls.line_accuracy(pred_lane, gt_lane, threshold) for gt_lane, threshold in zip(gt_lanes, thresholds)]
            max_accuracy = max(accuracies) if accuracies else 0.0
            if max_accuracy < cls.point_thresh:
                fp += 1.0

        if len(gt_lanes) > 4 and fn > 0:
            fn -= 1.0

        accuracy_sum = float(sum(line_accuracies))
        if len(gt_lanes) > 4 and line_accuracies:
            accuracy_sum -= float(min(line_accuracies))

        normalizer = max(min(4.0, float(len(gt_lanes))), 1.0)
        accuracy = accuracy_sum / normalizer
        fp_rate = fp / len(pred_lanes) if len(pred_lanes) > 0 else 0.0
        fn_rate = fn / normalizer

        return {'accuracy': accuracy, 'fp': fp_rate, 'fn': fn_rate}

    @classmethod
    def evaluate_dataset(cls, predictions, annotations):
        rows = []
        for pred, gt in zip(predictions, annotations):
            rows.append(cls.evaluate_image(pred['lanes'], gt['lanes'], gt['h_samples']))
        frame = pd.DataFrame(rows)
        return frame, {
            'accuracy': float(frame['accuracy'].mean()) if len(frame) else 0.0,
            'fp': float(frame['fp'].mean()) if len(frame) else 0.0,
            'fn': float(frame['fn'].mean()) if len(frame) else 0.0,
        }

## Inference and Lane Conversion

The model predicts lanes on resized input images. Before evaluation, each fitted polynomial is sampled at the original TuSimple `h_samples` and scaled back to the original image coordinate system.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def resolve_image_path(annotation):
    raw_file = annotation['raw_file']
    candidates = [
        TUSIMPLE_ROOT / raw_file,
        TUSIMPLE_ROOT / 'train_set' / raw_file,
        TUSIMPLE_ROOT / 'test_set' / raw_file,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not resolve image path for raw_file={raw_file}')


def preprocess_image_for_model(image_bgr):
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(image_rgb, (cfg.IMAGE_WIDTH, cfg.IMAGE_HEIGHT))
    x = resized.astype(np.float32) / 255.0
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).float()
    return x, image_rgb


def lanes_to_tusimple_points(lanes, h_samples, original_width, original_height):
    pred_lanes = []
    scale_x = original_width / cfg.IMAGE_WIDTH
    scale_y = cfg.IMAGE_HEIGHT / original_height

    for lane in lanes:
        lane_points = []
        for y_original in h_samples:
            y_resized = y_original * scale_y
            if y_resized < lane['y_min'] or y_resized > lane['y_max']:
                lane_points.append(-2)
                continue

            x_resized = float(np.polyval(lane['poly'], y_resized))
            x_original = int(round(x_resized * scale_x))

            if x_original < 0 or x_original >= original_width:
                lane_points.append(-2)
            else:
                lane_points.append(x_original)

        if sum(x >= 0 for x in lane_points) >= 2:
            pred_lanes.append(lane_points)

    return pred_lanes


@torch.no_grad()
def predict_one(annotation):
    image_path = resolve_image_path(annotation)
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        raise RuntimeError(f'Failed to read image: {image_path}')

    original_height, original_width = image_bgr.shape[:2]
    x, image_rgb = preprocess_image_for_model(image_bgr)
    x = x.to(DEVICE)

    start = time.perf_counter()
    binary_logits, embedding = model(x)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    runtime_ms = (time.perf_counter() - start) * 1000.0

    binary_np = binary_logits.squeeze(0).detach().cpu().numpy()
    embedding_np = embedding.squeeze(0).detach().cpu().numpy()
    supported_postprocess_keys = inspect.signature(my_postprocess).parameters.keys()
    postprocess_kwargs = {
        key: value
        for key, value in POSTPROCESS_KWARGS.items()
        if key in supported_postprocess_keys
    }
    lanes = my_postprocess(binary_np, embedding_np, **postprocess_kwargs)
    pred_lanes = lanes_to_tusimple_points(
        lanes,
        annotation['h_samples'],
        original_width=original_width,
        original_height=original_height,
    )

    prediction = {
        'lanes': pred_lanes,
        'h_samples': annotation['h_samples'],
        'raw_file': annotation['raw_file'],
        'run_time': runtime_ms,
    }
    return prediction, lanes, image_rgb

## Run Evaluation

In [ ]:
predictions = []
runtimes = []

for annotation in tqdm(annotations, desc='Testing TuSimple'):
    prediction, _, _ = predict_one(annotation)
    predictions.append(prediction)
    runtimes.append(prediction['run_time'])

per_image_metrics, summary_metrics = TuSimpleLaneEval.evaluate_dataset(predictions, annotations)
summary_metrics['num_samples'] = len(predictions)
summary_metrics['avg_runtime_ms'] = float(np.mean(runtimes)) if runtimes else 0.0
summary_metrics['fps'] = float(1000.0 / summary_metrics['avg_runtime_ms']) if summary_metrics['avg_runtime_ms'] > 0 else 0.0
summary_metrics['architecture'] = architecture
summary_metrics['checkpoint'] = str(CHECKPOINT_PATH)

summary_metrics

In [ ]:
print(f"TuSimple Accuracy: {summary_metrics['accuracy'] * 100:.2f}")
print(f"False Positive Rate: {summary_metrics['fp']:.4f}")
print(f"False Negative Rate: {summary_metrics['fn']:.4f}")
print(f"Average runtime: {summary_metrics['avg_runtime_ms']:.2f} ms/frame")
print(f"FPS: {summary_metrics['fps']:.2f}")

## Save Predictions and Results

In [ ]:
prediction_path = OUTPUT_DIR / 'tusimple_predictions.json'
summary_path = OUTPUT_DIR / 'tusimple_summary.json'
per_image_path = OUTPUT_DIR / 'tusimple_per_image_metrics.csv'

with open(prediction_path, 'w') as file:
    for prediction in predictions:
        file.write(json.dumps(prediction) + '\n')

with open(summary_path, 'w') as file:
    json.dump(summary_metrics, file, indent=2)

per_image_metrics.to_csv(per_image_path, index=False)

print('Saved predictions:', prediction_path)
print('Saved summary:', summary_path)
print('Saved per-image metrics:', per_image_path)

## Visual Check

Use this cell to verify that the model predictions and post-processing look reasonable before running the full dataset.

In [ ]:
import matplotlib.pyplot as plt

sample_index = 0
prediction, lanes, image_rgb = predict_one(annotations[sample_index])

# Draw lanes on the resized image used by the model for easier postprocess visualization.
resized_rgb = cv2.resize(image_rgb, (cfg.IMAGE_WIDTH, cfg.IMAGE_HEIGHT))
annotated = draw_lanes(resized_rgb, lanes, thickness=4)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title('Original image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(annotated)
plt.title(f"Predicted lanes: {len(prediction['lanes'])}")
plt.axis('off')
plt.show()

print('Raw file:', prediction['raw_file'])
print('Predicted lane count:', len(prediction['lanes']))
print('Ground truth lane count:', len(annotations[sample_index]['lanes']))

## Report Table Helper

In [ ]:
report_row = pd.DataFrame([
    {
        'Model': architecture,
        'Accuracy': summary_metrics['accuracy'] * 100,
        'FP': summary_metrics['fp'],
        'FN': summary_metrics['fn'],
        'FPS': summary_metrics['fps'],
    }
])
report_row